In [ ]:
# pulse_ps_only.py
# Full PS-only pipeline — no PL MLP classifier.
# Uses numpy MLP inference on ARM instead of sklearn pickle (avoids version mismatch).
# Use this to verify the full system works correctly before integrating PL.
#
# Pipeline:
#   USB camera → MoveNet TFLite (ARM) → numpy MLP (ARM) → rep counter → display
#
# Usage:
#   python3 pulse_ps_only.py
#
# Files needed on board:
#   /home/root/base.bit + base.hwh
#   /home/root/movenet_lightning.tflite
#   /home/root/coefs_0.npy, coefs_1.npy, coefs_2.npy
#   /home/root/intercepts_0.npy, intercepts_1.npy, intercepts_2.npy
#   /home/root/scaler_mean.npy, scaler_scale.npy
#   /home/root/pulse_ps_only.py
#   /home/root/rep_counter.py
#
# Generate .npy files on Mac first:
#   python3 export_numpy_weights.py

import cv2
import numpy as np
import time

# ── PYNQ imports ──────────────────────────────────────────────────────────────
try:
    from pynq.overlays.base import BaseOverlay
    from pynq.lib.button import Button
    from pynq.lib.video import *
    BOARD = True
except ImportError:
    print("[WARN] PYNQ not available — running in simulation mode")
    BOARD = False

try:
    import tflite_runtime.interpreter as tflite_mod
    def make_interpreter(path):
        return tflite_mod.Interpreter(path, num_threads=4)
except ImportError:
    import tensorflow as tf
    def make_interpreter(path):
        return tf.lite.Interpreter(model_path=path, num_threads=4)

from rep_counter_new import RepCounter, CLASS_NAMES

# ── Configuration ─────────────────────────────────────────────────────────────
BITSTREAM_PATH = "/home/root/base.bit"
TFLITE_PATH    = "./movenet_lightning.tflite"
MODEL_DIR      = "./"
CAMERA_INDEX   = 0
IMAGE_SIZE     = 192
CONF_THRESHOLD = 0.15

DEBOUNCE_TIME = 0
NUM_BUTTONS = 4

# MoveNet skeleton edges
EDGES = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]

# ── Load MoveNet ───────────────────────────────────────────────────────────────
print("Loading MoveNet...")
interpreter = make_interpreter(TFLITE_PATH)
interpreter.allocate_tensors()
inp_det = interpreter.get_input_details()
out_det = interpreter.get_output_details()

# ── Load MLP weights as numpy arrays (no sklearn/pickle dependency) ────────────
print("Loading MLP weights...")
coefs = [
    np.load(f"{MODEL_DIR}/coefs_0.npy"),
    np.load(f"{MODEL_DIR}/coefs_1.npy"),
    np.load(f"{MODEL_DIR}/coefs_2.npy"),
]
intercepts = [
    np.load(f"{MODEL_DIR}/intercepts_0.npy"),
    np.load(f"{MODEL_DIR}/intercepts_1.npy"),
    np.load(f"{MODEL_DIR}/intercepts_2.npy"),
]
scaler_mean  = np.load(f"{MODEL_DIR}/scaler_mean.npy")
scaler_scale = np.load(f"{MODEL_DIR}/scaler_scale.npy")
print(f"MLP: {coefs[0].shape[0]} → {coefs[0].shape[1]} → {coefs[1].shape[1]} → {coefs[2].shape[1]}")

# ── Inference ──────────────────────────────────────────────────────────────────
def run_movenet(frame):
    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (IMAGE_SIZE, IMAGE_SIZE))
    inp     = np.expand_dims(resized, axis=0).astype(np.uint8)
    interpreter.set_tensor(inp_det[0]['index'], inp)
    interpreter.invoke()
    return interpreter.get_tensor(out_det[0]['index']).reshape(17, 3)

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

def classify(keypoints):
    """numpy MLP forward pass — no sklearn needed."""
    x = (keypoints[:, :2].flatten() - scaler_mean) / scaler_scale
    x = np.maximum(x @ coefs[0] + intercepts[0], 0)   # layer 1 + ReLU
    x = np.maximum(x @ coefs[1] + intercepts[1], 0)   # layer 2 + ReLU
    x = x @ coefs[2] + intercepts[2]                   # output layer
    proba = softmax(x)
    cls   = int(np.argmax(proba))
    return cls, float(proba[cls])

# ── Stats tracking ─────────────────────────────────────────────────────────────
exercise_init = {str(i): 0 for i in range(len(CLASS_NAMES))}
previous_stats = []
exercise_max   = [None] * len(exercise_init)
player_num     = 1

class Stats:
    def __init__(self, name=""):
        global player_num
        self.name = f"Player {player_num}" if name == "" else name
        player_num += 1
        self.exercise_cnts = exercise_init.copy()

    def reset_count(self):
        self.exercise_cnts = exercise_init.copy()

    def set_exercise_cnts(self, d):
        self.exercise_cnts = d

    def to_string(self):
        parts = [f"{v}"
                 for k, v in self.exercise_cnts.items()]
        return f"{self.name}: " + "{" + ", ".join(parts) + "}"

    def copy(self):
        global player_num
        c = Stats(self.name)
        c.set_exercise_cnts(self.exercise_cnts.copy())
        player_num -= 1
        return c

# ── Display ────────────────────────────────────────────────────────────────────
def draw_display(frame, keypoints, exercise, exercise_index, confidence, reps, state, fps,
                 current_stat, prev_stats, ex_max, skeleton=True, is_recording=False):
    h, w = frame.shape[:2]

    # Skeleton
    if skeleton:
        for a, b in EDGES:
            if keypoints[a,2] > CONF_THRESHOLD and keypoints[b,2] > CONF_THRESHOLD:
                cv2.line(frame,
                         (int(keypoints[a,1]*w), int(keypoints[a,0]*h)),
                         (int(keypoints[b,1]*w), int(keypoints[b,0]*h)),
                         (0,255,0), 2)
        for k in keypoints:
            if k[2] > CONF_THRESHOLD:
                cv2.circle(frame, (int(k[1]*w), int(k[0]*h)), 4, (0,255,255), -1)


    # Top info panel
    cv2.rectangle(frame, (0,0), (320,175), (0,0,0), -1)
    col = (0,255,0) if confidence > 0.7 else (0,165,255) if confidence > 0.4 else (0,0,255)
    cv2.putText(frame, f"{exercise.upper()}",
                (10,40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, col, 2)
    cv2.putText(frame, f"conf: {confidence:.0%}",
                (10,65), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)
    cv2.putText(frame, f"REPS: {list(reps.values())}",
                (10,115), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)
    cv2.putText(frame, f"{state}",
                (10,145), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
    cv2.putText(frame, f"FPS: {fps:.1f}",
                (10,168), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (100,255,100), 1)

    # Bottom stats panel
    ix, iy = 0, 175
    panel_h = 40 + len(ex_max) * 38
    cv2.rectangle(frame, (ix,iy), (ix+320, iy+panel_h), (0,0,0), -1)
    last = prev_stats[-1].to_string() if prev_stats else "N/A"
    cv2.putText(frame, f"Last: {last}",
                (ix+10, iy+28), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)
    for i in range(len(ex_max)):
        label = CLASS_NAMES.get(i, "unknown")
        best  = f"{ex_max[i][0]}: {ex_max[i][1]}" if ex_max[i] else "N/A"
        cv2.putText(frame, f"Best {label}: {best}",
                    (ix+10, iy+28+(i+1)*36), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200,200,200), 1)
    
    if is_recording:
        cv2.putText(frame, f"RECORDING",
                    (ix+10, iy+28+(len(ex_max)+1)*36), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

    return frame

def get_time_ms():
    return time.time_ns() // 1_000_000

# ── Main ───────────────────────────────────────────────────────────────────────
def main():
    global previous_stats, exercise_max, player_num

    if BOARD:
        print(f"Loading bitstream: {BITSTREAM_PATH}")
        overlay     = BaseOverlay("base.bit")
        button_0    = overlay.buttons[0]
        button_1    = overlay.buttons[1]
        button_2    = overlay.buttons[2]
        button_3    = overlay.buttons[3]
        displayport = DisplayPort()
        displayport.configure(VideoMode(1920, 1080, 24), PIXEL_RGB)
        
        
         # Code taken from https://digilent.com/reference/learn/microprocessor/tutorials/debouncing-via-software/start?srsltid=AfmBOopyAjinABd9VAqmMSGY8QmHhWQSOWYKDYIcPH9McAYU_J87vzom
        from enum import Enum

        global DEBOUNCE_TIME, NUM_BUTTONS

        class Button(Enum):
            LOW = 0
            HIGH = 1

        currentBtnState = [Button.LOW] * NUM_BUTTONS
        prevBtnState = [Button.LOW] * NUM_BUTTONS
        currentActiveState = [Button.LOW] * NUM_BUTTONS
        prevActiveState = [Button.LOW] * NUM_BUTTONS

        lastDebounceTime = [get_time_ms()] * NUM_BUTTONS

        def get_button(i):
            currentBtnState[i] = Button.HIGH if (overlay.buttons[i].read()) else Button.LOW
            if (currentBtnState[i] != prevBtnState[i]):
                lastDebounceTime[i] = get_time_ms()
            if (get_time_ms() - lastDebounceTime[i]) > DEBOUNCE_TIME:
                currentActiveState[i] = currentBtnState[i]
            temp_return = (prevActiveState[i] == Button.LOW and currentActiveState[i] == Button.HIGH)
            prevBtnState[i] = currentBtnState[i]
            prevActiveState[i] = currentActiveState[i]
            if temp_return:
                print(f"Button {i} pressed")
            return temp_return
        

    counter      = RepCounter()
    currentStat  = Stats()
    is_recording = False
    skeleton = True

    cap = cv2.VideoCapture(CAMERA_INDEX)
    time.sleep(1.0)

    if not cap.isOpened():
        print("ERROR: Cannot open camera")
        return

    print("Running")
    if not BOARD:
        print("Q=quit  R=reset")

    prev_time = time.time()
    fps = 0.0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                time.sleep(0.05)
                continue

            keypoints            = run_movenet(frame)
            exercise_class, conf = classify(keypoints)
            reps, state, angle   = counter.update(exercise_class, keypoints, conf)
            exercise             = CLASS_NAMES.get(exercise_class, "unknown")

            if is_recording and counter.current_class is not None:
                currentStat.exercise_cnts[str(counter.current_class)] = counter.rep_count[counter.current_class]

            now = time.time()
            fps = 0.9 * fps + 0.1 / max(now - prev_time, 1e-6)
            prev_time = now

            frame = draw_display(frame, keypoints, exercise, counter.current_class, conf, reps, state, fps,
                                 currentStat, previous_stats, exercise_max, skeleton, is_recording)

            if BOARD:
                dp_frame = displayport.newframe()
                dp_frame[:] = cv2.resize(frame, (1920,1080))
                displayport.writeframe(dp_frame)

                if get_button(0):
                    counter.reset()
                    currentStat.reset_count()
                    for i in range(len(exercise_max)):
                        if exercise_max[i] and exercise_max[i][0] == currentStat.name:
                            exercise_max[i] = None

                if get_button(1):
                    if not is_recording:
                        is_recording = True
                        counter.reset()
                    else:
                        previous_stats.append(currentStat.copy())
                        for i in range(len(exercise_max)):
                            cnt = currentStat.exercise_cnts[str(i)]
                            if exercise_max[i] is None or exercise_max[i][1] < cnt:
                                exercise_max[i] = (currentStat.name, cnt)
                        currentStat  = Stats()
                        is_recording = False
                        counter.reset()

                if get_button(2):
                    skeleton = not(skeleton)

                if get_button(3):
                    previous_stats = []
                    player_num     = 1
                    currentStat    = Stats()
                    exercise_max   = [None] * len(exercise_init)
                    counter.reset()

            else:
                cv2.imshow("Pulse", frame)
                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'):
                    break
                elif key == ord('r'):
                    counter.reset()
                    currentStat.reset_count()
                    print("Reset")

    except KeyboardInterrupt:
        print("Stopped")
    finally:
        cap.release()
        if BOARD:
            displayport.close()
        cv2.destroyAllWindows()
        print(f"Session ended. Total reps: {counter.rep_count}")
        print("Final stats:")
        for s in previous_stats:
            print(s.to_string())

if __name__ == "__main__":
    main()


Loading MoveNet...
Loading MLP weights...
MLP: 34 → 128 → 64 → 3
Loading bitstream: /home/root/base.bit


[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (616) isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created


Running
Button 1 pressed
Button 1 pressed


In [4]:
!cat /sys/class/drm/card0-DP-1/modes

3840x2160
3840x2160
3840x2160
2560x1440
2048x1280
2048x1152
1920x1200
1920x1080
1920x1080
1920x1080
1920x1080
1920x1080
1920x1080
1600x1200
1680x1050
1280x1024
1280x1024
1280x800
1152x864
1280x720
1280x720
1280x720
1024x768
1024x768
800x600
800x600
720x576
720x576
640x480
640x480
640x480
640x480
720x400


In [7]:
!dmesg | grep -i drm

[    2.978044] xlnx-drm xlnx-drm.0: bound fd4a0000.display (ops 0xffff8000812180a8)
[    4.063566] zynqmp-display fd4a0000.display: [drm] Cannot find any crtc or sizes
[    4.071304] [drm] Initialized xlnx 1.0.0 20130509 for fd4a0000.display on minor 0
[    5.199581] zynqmp-display fd4a0000.display: [drm] Cannot find any crtc or sizes
[    8.939437] systemd[1]: Starting Load Kernel Module drm...
[   10.865769] [drm] Probing for xlnx,zocl
[   10.955442] zocl-drm axi:zyxclmm_drm: error -ENXIO: IRQ index 8 not found
[   10.987895] [drm] FPGA programming device pcap founded.
[   10.991182] [drm] PR[0] Isolation addr 0x0
[   11.015998] [drm] Initialized zocl 2.17.0 20250828 for axi:zyxclmm_drm on minor 1
[   18.646079] zocl-drm axi:zyxclmm_drm: zocl_create_client: created KDS client for pid(681), ret: 0
[   18.646131] zocl-drm axi:zyxclmm_drm: zocl_destroy_client: client exits pid(681)
[   19.967348] zocl-drm axi:zyxclmm_drm: zocl_create_client: created KDS client for pid(681), ret: 0
[   1

In [3]:
!v4l2-ctl -d /dev/video0 --list-ctrls

/bin/bash: line 1: v4l2-ctl: command not found


In [7]:
!sudo -y apt-get install v4l-utils

sudo: invalid option -- 'y'
usage: sudo -h | -K | -k | -V
usage: sudo -v [-ABknS] [-g group] [-h host] [-p prompt] [-u user]
usage: sudo -l [-ABknS] [-g group] [-h host] [-p prompt] [-U user] [-u user]
            [command]
usage: sudo [-ABbEHknPS] [-r role] [-t type] [-C num] [-D directory] [-g group]
            [-h host] [-p prompt] [-R directory] [-T timeout] [-u user]
            [VAR=value] [-i|-s] [<command>]
usage: sudo -e [-ABknS] [-r role] [-t type] [-C num] [-D directory] [-g group]
            [-h host] [-p prompt] [-R directory] [-T timeout] [-u user] file ...
